# Chương 3. HUẤN LUYỆN VÀ ĐÁNH GIÁ MÔ HÌNH

## Thuật Toán Tìm Cực Trị

### 1. Thuật Toán Gradient Descent
Gradient Descent (viết tắt là GD) là giải thuật đi tìm cực trị cho hàm mục tiêu.  Ý tưởng của phương pháp này là dò tìm điểm cực trị của hàm mất mát thông qua việc tính đạo hàm. Ở mỗi bước lặp, mô hình cập nhật các tham số dựa trên giá trị gradient. Quá trình này được lặp lại cho đến khi hàm mất mát dần tiến về $0$.
\begin{equation}
  \theta_{j+1} = \theta_j - \alpha \cdot \frac{\partial}{\partial \theta_j} \mathcal{L}(\theta) 
\end{equation}
Trong đó: 
- $\theta_j$ là giá trị tham số tại bước $j$.
- $\alpha$ là tốc độ học (learning rate).
- $\frac{\partial}{\partial \theta_j} \mathcal{L}(\theta)$ đạo hàm hàm mục tiêu $\mathcal{L}(\theta)$ theo $\theta_j$.

<p align="center">
  <img src="picture/gd_parallel_convergence.gif" width="800">
  <br>
  <em>Hình 1. Quá trình tối ưu bằng giải thuật Gradient Descent: mô hình tuyến tính (trái) và hàm mất mát hội tụ (phải). </em>
</p>


In [37]:
import numpy as np
import pandas as pd

# Tạo bộ dữ liệu mô phỏng
np.random.seed(42)  
n_samples = 100  

area = np.random.uniform(0, 10, n_samples) 
epsilon = np.random.normal(0, 1, n_samples)  
price = 2 * area + 3 + epsilon 

# Cài đặt tham số
alpha = 0.03  # Learning rate
iterations = 1000  # Số lượng vòng lặp

# Khởi tạo tham số huấn luyện
theta_0 = 0  # Intercept
theta_1 = 0  # Slope
n = len(area) 


# Gradient Descent
for iteration in range(1, iterations + 1):
    y_pred = theta_0 + theta_1 * area  # Dự đoán
    loss = (1 / (2 * n)) * np.sum((price - y_pred) ** 2) # Tính giá trị loss 
    
    # Tính gradient
    d_theta_0 = -(1 / n) * np.sum(price - y_pred)
    d_theta_1 = -(1 / n) * np.sum((price - y_pred) * area)
    
    # Cập nhật tham số huấn luyện
    theta_0 -= alpha * d_theta_0
    theta_1 -= alpha * d_theta_1
    
    # In kết quá sau mỗi lần lặp
    if iteration % 200 == 0 or iteration == 1:  
        print(f"Iteration {iteration}:")
        print(f"  Loss = {loss:.4f}")
        print(f"  theta_0 = {theta_0:.4f}, theta_1 = {theta_1:.4f}")

Iteration 1:
  Loss = 94.0413
  theta_0 = 0.3721, theta_1 = 2.2630
Iteration 200:
  Loss = 0.4447
  theta_0 = 2.6796, theta_1 = 2.0363
Iteration 400:
  Loss = 0.4048
  theta_0 = 3.1144, theta_1 = 1.9695
Iteration 600:
  Loss = 0.4033
  theta_0 = 3.1962, theta_1 = 1.9569
Iteration 800:
  Loss = 0.4033
  theta_0 = 3.2115, theta_1 = 1.9546
Iteration 1000:
  Loss = 0.4033
  theta_0 = 3.2144, theta_1 = 1.9541


### 2. Thuật toán Gradient Descent và Momentum
#### 2.1 Động lượng
Khái niệm động lượng (momentum) trong vật lý là một đại lượng vectơ mô tả khả năng truyền chuyển động của vật đang di chuyển được xác định bằng $\vec{p} = m \cdot \vec{v}$ với $m$ là khối lượng của vật và $\vec{v}$ là vận tốc của vật đang di chuyển.

<p align="center">
  <img src="picture/momentum simulator.gif" width="400">
  <br>
  <em>Hình 2. Newton’s Cradle – minh họa hoàn hảo cho bảo toàn động lượng và năng lượng trong va chạm đàn hồi.</em>
</p>



#### 2.2 Phương pháp trung bình động (moving average) 
Phương pháp trung bình động (moving average) là trung bình hàm mũ EMA (Exponential Moving Average) để các gradient được cập nhật trở nên mượt mà hơn khi huấn luyện. Công thức để xác định giá trị làm mượt $s(t)$ bằng phương pháp EMA tại thời điểm $t$ được xác định bởi:
$$
s(t) =
\begin{cases}
Y(1) & \text{khi } t = 1 \\
\beta \cdot s(t-1) + (1 - \beta) \cdot Y(t) & \text{khi } t > 1
\end{cases}
$$

Trong đó:
- $Y(t)$ là giá trị thực tế $Y$ quan sát tại thời điểm $t$.
- $\beta$ là hệ số làm mượt nằm trong nửa đoạn $[0,1]$.
- $t$ là điểm thời gian quan sát.

#### 2.3 Gradient kết hợp Momentum
Công thức Gradient Descent kết hợp momentum như sau:
$$
\begin{cases}
v_{j} = \beta v_{j-1} + (1 - \beta)\frac{\partial}{\partial \theta_j} \mathcal{L}(\theta) \\
\theta_{j+1} = \theta_j - \alpha \cdot v_j
\end{cases}
$$

Trong đó:
- $v_t$ giá trị Gradient đã được làm mượt.
- $\frac{\partial}{\partial \theta_j} \mathcal{L}(\theta)$ gradient của hàm mục tiêu $\mathcal{L}(\theta)$ tại $\theta_j$.
- $\beta$ là hệ số làm mượt nằm trong đoạn $[0,1]$.
- $\theta_j$ là giá trị tham số tại bước $j$.
- $\alpha$ là tốc độ học.

Công thức GD, các giá trị tham số $\theta_j$ chỉ được cập nhật dựa trên giá trị gradient tại một điểm $\theta_{j-1}$ thì GD kết hợp với momentum cho phép ghi nhớ toàn bộ các gradient của các $\theta$ trước đó từ $v_j$ (đã được làm mượt). Phương pháp này giúp duy trì hướng chuyển động hay nói cách khác thêm vào lực đẩy từ các gradient trước làm cho quá trình huấn luyện trở nên mượt mà hơn và nhanh hội tụ về điểm cực trị hơn.

In [29]:
# Cài đặt tham số
alpha = 0.03  # Learning rate
beta = 0.9  # Momentum 
iterations = 1000  


theta_0 = 0  # Intercept
theta_1 = 0  # Slope
v_theta_0 = 0  # vận tốc khởi tạo của theta_0
v_theta_1 = 0  # vận tốc khởi tạo của theta_1
n = len(area) 

# Gradient Descent kết hợp momentum
for iteration in range(1, iterations + 1):
    y_pred = theta_0 + theta_1 * area  
    loss = (1 / (2 * n)) * np.sum((price - y_pred) ** 2)  
    
    # Tính gradients
    d_theta_0 = -(1 / n) * np.sum(price - y_pred)
    d_theta_1 = -(1 / n) * np.sum((price - y_pred) * area)
    
    # Cập nhật velocities
    v_theta_0 = beta * v_theta_0 + (1 - beta) * d_theta_0
    v_theta_1 = beta * v_theta_1 + (1 - beta) * d_theta_1
    
    # Cập nhật tham số huấn luyện với momentum
    theta_0 -= alpha * v_theta_0
    theta_1 -= alpha * v_theta_1
    
    # Hiển thị kết quả
    if iteration % 200 == 0 or iteration == 1: 
        print(f"Iteration {iteration}:")
        print(f"  Loss = {loss:.4f}")
        print(f"  theta_0 = {theta_0:.4f}, theta_1 = {theta_1:.4f}")

Iteration 1:
  Loss = 94.0413
  theta_0 = 0.0372, theta_1 = 0.2263
Iteration 200:
  Loss = 0.4404
  theta_0 = 2.7086, theta_1 = 2.0318
Iteration 400:
  Loss = 0.4043
  theta_0 = 3.1332, theta_1 = 1.9666
Iteration 600:
  Loss = 0.4033
  theta_0 = 3.2018, theta_1 = 1.9561
Iteration 800:
  Loss = 0.4033
  theta_0 = 3.2130, theta_1 = 1.9544
Iteration 1000:
  Loss = 0.4033
  theta_0 = 3.2147, theta_1 = 1.9541


### 3. Thuật toán Stochastic Gradient Descent
Stochastic Gradient Descent (viết tắt là SGD) nhằm cải tiến Gradient Descent thông thường bằng cách kết hợp kỹ thuật lấy mẫu (sampling) ngẫu nhiên trong quá trình tính toán gradient. Thay vì tính gradient dựa trên toàn bộ tập dữ liệu, SGD chỉ sử dụng một phần nhỏ các mẫu hoặc thậm chí một điểm dữ liệu duy nhất trong mỗi bước lặp để tính toán gradient.

$$
\theta_{j+1} = \theta_j - \alpha \cdot \frac{\partial}{\partial \theta_j} \mathcal{L}(\theta; x^{(i)}, y^{(i)}) 
$$
Trong đó:
- $\theta_j$ là giá trị tham số tại bước lặp thứ $j$.
- $\alpha$: là tốc độ học.
- $i$ chỉ số của mẫu được chọn ngẫu nhiên từ tập dữ liệu.
- $\mathcal{L}(\theta; x^{(i)}, y^{(i)})$ là hàm mất mát tính trên một mẫu ngẫu nhiên $(x^{(i)}, y^{(i)})$.
- $\theta_{j+1}$ tham số sau khi cập nhật.

In [ ]:
import numpy as np
import pandas as pd


# Cài đặt tham số
alpha = 0.03  # Tốc độ học
iterations = 1000  # Số lần lặp

# Khởi tạo tham số huấn luyện
theta_0 = 0  # Intercept
theta_1 = 0  # Slope

# Stochastic Gradient Descent
for iteration in range(iterations):
    # Chọn ngẫu nhiên 10 điểm dữ liệu
    indices = np.random.permutation(10)  
    area_shuffled = area[indices]
    price_shuffled = price[indices]

    for i in range(10):
        x_i = area_shuffled[i]  
        y_i = price_shuffled[i]  

        # Dự đoán
        y_pred = theta_0 + theta_1 * x_i

        # Tính giá trị gradient
        d_theta_0 = -(y_i - y_pred)
        d_theta_1 = -(y_i - y_pred) * x_i

        # Cập nhật tham số
        theta_0 -= alpha * d_theta_0
        theta_1 -= alpha * d_theta_1

    y_pred_all = theta_0 + theta_1 * area
    loss = (1 / (2 * n_samples)) * np.sum((price - y_pred_all) ** 2)
    
 # In kết quá sau mỗi lần lặp
    if iteration % 200 == 0 or iteration == 1:  
        print(f"Iteration {iteration}:")
        print(f"  Loss = {loss:.4f}")
        print(f"  theta_0 = {theta_0:.4f}, theta_1 = {theta_1:.4f}")

Iteration 0:
  Loss = 1.7170
  theta_0 = 0.5906, theta_1 = 2.2062
Iteration 1:
  Loss = 3.2205
  theta_0 = 0.7989, theta_1 = 1.9631
Iteration 200:
  Loss = 0.7698
  theta_0 = 3.3460, theta_1 = 1.7805
Iteration 400:
  Loss = 0.4873
  theta_0 = 3.4611, theta_1 = 1.9864
Iteration 600:
  Loss = 2.6237
  theta_0 = 3.5204, theta_1 = 1.5294
Iteration 800:
  Loss = 7.0570
  theta_0 = 3.5846, theta_1 = 2.5534


### 4. Thuật toán Adaptive Gradient Descent
AdaGrad là một phương pháp tối ưu giúp tự điều chỉnh tốc độ học cho từng lần cập nhật dựa trên lịch sử gradient. Bằng cách sử dụng thông tin về độ lớn và hướng của các gradient trước đó, AdaGrad khắc phục hạn chế của Gradient Descent truyền thống vốn dùng learning rate cố định và dễ mắc kẹt trong các điểm tối ưu cục bộ. Phương pháp này giúp quá trình cập nhật trọng số trở nên linh hoạt và hiệu quả hơn.

\begin{equation}
\begin{cases}
G_j = G_{j-1} + g_j^2, \\[6pt]
\theta_{j+1} = \theta_j - \dfrac{\alpha}{\sqrt{G_j + \epsilon}} \cdot g_j
\end{cases}
\end{equation}
Trong đó: 
- $\alpha$ là giá trị hằng số.
- $g_j$ là gradient tại bước hiện tại.
- $G_j$ là tổng bình phương gradient qua các bước lặp.
- $\epsilon$ giá trị hằng số rất nhỏ được thêm vào để đảm bảo mẫu số khác 0.


In [43]:

# Cài đặt tham số
alpha = 0.5       # Tốc độ học ban đầu
iterations = 1000   # Số lần lặp
epsilon = 1e-8      # Tránh chia cho 0

# Khởi tạo tham số huấn luyện
theta_0 = 0  # Intercept
theta_1 = 0  # Slope

# Khởi tạo tổng bình phương gradient (đặc trưng của AdaGrad)
G0 = 0
G1 = 0

# AdaGrad (Full-Batch)
for iteration in range(iterations):

    # Dự đoán
    y_pred = theta_0 + theta_1 * area

    # Gradient (full dataset)
    d_theta_0 = -np.sum(price - y_pred)
    d_theta_1 = -np.sum((price - y_pred) * area)

    # Tích lũy bình phương gradient
    G0 += d_theta_0**2
    G1 += d_theta_1**2

    # Tốc độ học thích nghi cho từng tham số
    adaptive_lr_0 = alpha / (np.sqrt(G0) + epsilon)
    adaptive_lr_1 = alpha / (np.sqrt(G1) + epsilon)

    # Cập nhật tham số
    theta_0 -= adaptive_lr_0 * d_theta_0
    theta_1 -= adaptive_lr_1 * d_theta_1

    # Tính loss toàn bộ
    y_pred_all = theta_0 + theta_1 * area
    loss = (1 / (2 * n_samples)) * np.sum((price - y_pred_all) ** 2)

    # In kết quả
    if iteration % 200 == 0 or iteration == 1:
        print(f"Iteration {iteration}:")
        print(f"  Loss = {loss:.6f}")
        print(f"  theta_0 = {theta_0:.6f}, theta_1 = {theta_1:.6f}")


Iteration 0:
  Loss = 55.281815
  theta_0 = 0.500000, theta_1 = 0.500000
Iteration 1:
  Loss = 36.769833
  theta_0 = 0.805080, theta_1 = 0.803606
Iteration 200:
  Loss = 0.422067
  theta_0 = 2.852506, theta_1 = 2.012002
Iteration 400:
  Loss = 0.405228
  theta_0 = 3.098683, theta_1 = 1.972637
Iteration 600:
  Loss = 0.403492
  theta_0 = 3.177711, theta_1 = 1.960000
Iteration 800:
  Loss = 0.403313
  theta_0 = 3.203090, theta_1 = 1.955942


### 5. Thuật toán Root Mean Square Propagation
Root Mean Square Propagation (viết tắt là RMSprop) được phát triển nhằm khắc phục hạn chế của AdaGrad, đặc biệt là vấn đề tốc độ học (learning rate) giảm quá nhanh khi số lần cập nhật tăng lên. Trong khi AdaGrad lưu trữ toàn bộ lịch sử gradient để tính tổng bình phương gradient, RMSprop sử dụng một cách tiếp cận hiệu quả hơn, thay vì tính tổng, thuật toán áp dụng giá trị trung bình động theo cấp số nhân (exponential moving average) của bình phương gradient. 

\begin{equation}
\begin{cases}
E[g^2]_j = \beta E[g^2]_{j-1} + (1 - \beta) g_j^2, \\[8pt]
\theta_{j+1} = \theta_j - \dfrac{\alpha}{\sqrt{E[g^2]_j + \epsilon}} \cdot g_j
\end{cases}
\end{equation}

Trong đó:
- $\theta_j$: Giá trị tham số của mô hình tại bước lặp thứ $j$.
- $g_j = \frac{\partial}{\partial \theta_j} \mathcal{L}(\theta) $: Gradient của hàm mất mát theo tham số $\theta$ tại bước $j$.
 $E[g^2]_j$: Giá trị trung bình động (exponential moving average) của bình phương gradient tại bước lặp thứ $j$.
- $\beta$: Hệ số suy giảm (decay rate) được chọn trong khoảng $0.9 \le \beta \le 0.999$.
- $\alpha$: Tốc độ học.
- $\epsilon$: Hằng số rất nhỏ được thêm vào để mẫu số luôn khác 0.
- $\theta_{j+1}$: Giá trị tham số sau khi đã cập nhật tại bước $j+1$.


In [ ]:
# Cài đặt tham số
alpha = 0.03        # Tốc độ học (learning rate)
iterations = 1000   # Số lần lặp
beta = 0.9          # Hệ số làm mượt (decay rate)
epsilon = 1e-8      # Tránh chia cho 0

# Khởi tạo tham số huấn luyện
theta_0 = 0   # Intercept
theta_1 = 0   # Slope

# Khởi tạo E[g^2] cho RMSProp
Eg2_0 = 0
Eg2_1 = 0

# RMSProp
for iteration in range(iterations):

    # Dự đoán
    y_pred = theta_0 + theta_1 * area

    # Tính gradient toàn bộ dữ liệu
    d_theta_0 = -np.sum(price - y_pred)
    d_theta_1 = -np.sum((price - y_pred) * area)

    # Cập nhật bình phương gradient theo EMA (Exponential Moving Average)
    Eg2_0 = beta * Eg2_0 + (1 - beta) * (d_theta_0**2)
    Eg2_1 = beta * Eg2_1 + (1 - beta) * (d_theta_1**2)

    # Tính learning rate hiệu chỉnh RMSProp
    adaptive_lr_0 = alpha / (np.sqrt(Eg2_0) + epsilon)
    adaptive_lr_1 = alpha / (np.sqrt(Eg2_1) + epsilon)

    # Cập nhật tham số
    theta_0 -= adaptive_lr_0 * d_theta_0
    theta_1 -= adaptive_lr_1 * d_theta_1

    # Tính loss toàn bộ
    y_pred_all = theta_0 + theta_1 * area
    loss = (1 / (2 * n_samples)) * np.sum((price - y_pred_all) ** 2)

    # In kết quả
    if iteration % 200 == 0 or iteration == 1:
        print(f"Iteration {iteration}:")
        print(f"  Loss = {loss:.6f}")
        print(f"  theta_0 = {theta_0:.6f}, theta_1 = {theta_1:.6f}")

Iteration 0:
  Loss = 85.894087
  theta_0 = 0.094868, theta_1 = 0.094868
Iteration 1:
  Loss = 80.338079
  theta_0 = 0.162221, theta_1 = 0.162182
Iteration 200:
  Loss = 0.413801
  theta_0 = 3.236466, theta_1 = 1.976781
Iteration 400:
  Loss = 0.407935
  theta_0 = 3.230096, theta_1 = 1.969023
Iteration 600:
  Loss = 0.407935
  theta_0 = 3.230096, theta_1 = 1.969023
Iteration 800:
  Loss = 0.407935
  theta_0 = 3.230096, theta_1 = 1.969023


### 6. Thuật toán Adaptive Moment Estimation 
Adaptive Moment Estimation (viết tắt là Adam) là sự kết hợp giữa thuật toán RMSprop và momentum. Nghĩa là Adam không chỉ sử dụng trung bình động của bình phương gradient mà còn kết hợp với trung bình động của chính gradient. Điều này mang lại sự cân bằng giữa tốc độ học nhanh và tính ổn định trong quá trình tối ưu hóa, khiến Adam trở thành lựa chọn phổ biến trong huấn luyện các mô hình học sâu.

Bước 1: Trung bình động bậc nhất (momentum)
\begin{equation}
  m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t
\end{equation}
Trong đó:
- $m_t$ là trung bình động bậc nhất của gradient tại thời điểm $t$.

- $g_t$ là gradient của hàm mất mát đối với tham số tại thời điểm $t$.

- $\beta_1$ hệ số giảm bậc nhất, thường được chọn là 0.9 giúp kiểm soát độ lớn của gradient trước đó bằng cách cập nhật giá trị mới của $m_t$.

- $m_{t-1}$ là trung bình động bậc nhất tại thời điểm $t-1$.


Bước 2: Trung bình động bậc hai

\begin{equation}
  v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2
\end{equation}    
Trong đó:
  - $v_t$ là trung bình động bậc hai của gradient tại thời điểm $t$.    
  - $g_t$ là gradient của hàm mất mát đối với tham số tại thời điểm $t$.    
  - $\beta_2$ hệ số giảm bậc hai, thường được chọn là $0.999$, dùng để kiểm soát mức độ làm mịn (smoothing) của trung bình động bậc hai.    
  - $v_{t-1}$ là trung bình động bậc hai tại thời điểm $t-1$.


Bước 3: Hiệu chỉnh thiên lệch (bias correction)

Để khắc phục vấn đề ban đầu của trung bình động (thiên lệch về 0), Adam áp dụng hiệu chỉnh như sau:

$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$
    
Bước 4: Cập nhật tham số

\begin{equation}
     \theta_{t+1} = \theta_t - \alpha \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}
\end{equation}
Trong đó:  

- $\alpha$ là tốc độ học.
- $\epsilon$ là một hằng số rất nhỏ để đảm bảo mẫu số khác $0$.


In [ ]:
# Cài đặt tham số
alpha = 0.03         # Tốc độ học (learning rate)
iterations = 1000    # Số lần lặp
beta1 = 0.9          # Hệ số mô-men bậc 1 (momentum term)
beta2 = 0.999        # Hệ số mô-men bậc 2 (RMSProp term)
epsilon = 1e-8       # Tránh chia cho 0

# Khởi tạo tham số
theta_0 = 0
theta_1 = 0

# Khởi tạo mô-men cho Adam
m0, v0 = 0, 0   # mô-men bậc 1 và bậc 2 của theta_0
m1, v1 = 0, 0   # mô-men bậc 1 và bậc 2 của theta_1

# Adam (Full-Batch)
for iteration in range(iterations):

    # Dự đoán
    y_pred = theta_0 + theta_1 * area

    # Gradient toàn bộ dữ liệu
    g0 = -np.sum(price - y_pred)
    g1 = -np.sum((price - y_pred) * area)

    # ---- Tính mô-men bậc 1 (momentum) ----
    m0 = beta1 * m0 + (1 - beta1) * g0
    m1 = beta1 * m1 + (1 - beta1) * g1

    # ---- Tính mô-men bậc 2 (RMSProp) ----
    v0 = beta2 * v0 + (1 - beta2) * (g0**2)
    v1 = beta2 * v1 + (1 - beta2) * (g1**2)

    # ---- Hiệu chỉnh bias (bias-correction) ----
    m0_hat = m0 / (1 - beta1**(iteration + 1))
    m1_hat = m1 / (1 - beta1**(iteration + 1))

    v0_hat = v0 / (1 - beta2**(iteration + 1))
    v1_hat = v1 / (1 - beta2**(iteration + 1))

    # ---- Cập nhật tham số ----
    theta_0 -= alpha * m0_hat / (np.sqrt(v0_hat) + epsilon)
    theta_1 -= alpha * m1_hat / (np.sqrt(v1_hat) + epsilon)

    # Tính loss toàn bộ
    y_pred_all = theta_0 + theta_1 * area
    loss = (1 / (2 * n_samples)) * np.sum((price - y_pred_all)**2)

    # In kết quả
    if iteration % 200 == 0 or iteration == 1:
        print(f"Iteration {iteration}:")
        print(f"  Loss = {loss:.6f}")
        print(f"  theta_0 = {theta_0:.6f}, theta_1 = {theta_1:.6f}")


Iteration 0:
  Loss = 91.424776
  theta_0 = 0.030000, theta_1 = 0.030000
Iteration 1:
  Loss = 88.846394
  theta_0 = 0.059988, theta_1 = 0.059988
Iteration 200:
  Loss = 0.521734
  theta_0 = 2.305140, theta_1 = 2.100395
Iteration 400:
  Loss = 0.453411
  theta_0 = 2.622712, theta_1 = 2.048792
Iteration 600:
  Loss = 0.418676
  theta_0 = 2.886902, theta_1 = 2.006526
Iteration 800:
  Loss = 0.406768
  theta_0 = 3.059094, theta_1 = 1.978979


## Tổng kết

<p align="center">
  <img src="picture/optimizers_with_minibatch.gif" width="900">
  <br>
  <em>Hình 3. Ví dụ mô phỏng so sánh quá trình tối ưu các thuật toán tối trên hàm số f(x,y) = x^2+10y^2.</em>
</p>
